# Module 4 - Topic 3: The ML Workflow
## Live Demo Notebook

**Scenario:** A Nigerian school wants to identify students at risk of failing their WAEC.
We have 100 student records with study hours, attendance, past scores, and whether they passed.

We'll follow the **complete 5-step ML workflow** - one step at a time.

---

**Step 1:** Define the problem  
**Step 2:** Collect and prepare data  
**Step 3:** Split the data  
**Step 4:** Train the model  
**Step 5:** Evaluate and iterate  

---

---
## Step 1 - Define the Problem

**Business question:** Which students are at risk of failing their WAEC?

**ML framing:** Binary classification  
- Input: study_hours, attendance_pct, prev_score  
- Output: passed (1) or failed (0)  

**Success metric:** Correctly identify at least 80% of students (accuracy ≥ 0.80)  

**Constraint:** Model must be explainable - school management needs to understand why a student is flagged.

---
## Step 2 - Collect and Prepare Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load the data
df = pd.read_csv('student_scores.csv')
print('Shape:', df.shape)
df.head(8)

In [ ]:
# Quick EDA - always before modelling
print('Missing values:', df.isnull().sum().sum())
print('Pass rate:', df['passed'].mean())
print()
print('Feature statistics:')
print(df.describe().round(2))

In [ ]:
# Feature engineering - create more informative features
df['attended_enough'] = (df['attendance_pct'] >= 75).astype(int)  # binary flag
df['effort_score']    = df['study_hours'] * df['prev_score'] / 100  # interaction term

print('After feature engineering, columns:')
print(df.columns.tolist())
print()
df.head(5)

In [ ]:
# Select features and target
FEATURES = ['study_hours', 'attendance_pct', 'prev_score', 'attended_enough', 'effort_score']
TARGET   = 'passed'

X = df[FEATURES]
y = df[TARGET]

print('Features:')
for f in FEATURES:
    print(f'  {f}')
print(f'\nTarget: {TARGET} ({y.sum()} passed / {(y==0).sum()} failed out of {len(y)})')

---
## Step 3 - Split the Data

**Rule: NEVER evaluate on training data.**
Hold back 20% as a test set - the model will never see these rows during training.

In [ ]:
# 80% training, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set: {len(X_train)} students')
print(f'Test set:     {len(X_test)} students')
print()
print(f'Pass rate in training set: {y_train.mean():.1%}')
print(f'Pass rate in test set:     {y_test.mean():.1%}')
print()
print('The test set is now locked - we will only look at it ONCE for final evaluation.')

---
## Step 4 - Train the Model

In [ ]:
# Create and train the model
model = DecisionTreeClassifier(max_depth=5, random_state=42)
model.fit(X_train, y_train)  # learning happens here

# Training accuracy (not the real measure - just a sanity check)
train_acc = model.score(X_train, y_train)
print(f'Training accuracy: {train_acc:.2%}')
print()
print('Training done. The model has never seen the test set.')
print('max_depth=5 means the model can ask at most 5 questions to make a decision.')

---
## Step 5 - Evaluate and Iterate

In [ ]:
# Predict on the test set (first and only look)
y_pred = model.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)

print(f'Training accuracy: {train_acc:.2%}')
print(f'Test accuracy:     {test_acc:.2%}')
print()
gap = train_acc - test_acc
if gap > 0.10:
    print(f'Gap: {gap:.2%} - large gap suggests overfitting. Investigate in Topic 4.')
elif gap > 0.05:
    print(f'Gap: {gap:.2%} - moderate gap. Some overfitting, acceptable for now.')
else:
    print(f'Gap: {gap:.2%} - small gap. Good generalisation.')

In [ ]:
# Detailed performance breakdown
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Failed', 'Passed']))

In [ ]:
# What did the model learn to pay attention to?
importance = pd.Series(
    model.feature_importances_,
    index=FEATURES
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 4))
importance.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Feature Importance - What the Model Learned Matters Most',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()
print('Higher bar = more important to the model\'s predictions.')

In [ ]:
# Iteration: try without the engineered features - does the model do better or worse?
X_base = df[['study_hours', 'attendance_pct', 'prev_score']]
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_base, y, test_size=0.2, random_state=42
)

baseline_model = DecisionTreeClassifier(max_depth=5, random_state=42)
baseline_model.fit(X_train_b, y_train_b)
baseline_acc = accuracy_score(y_test_b, baseline_model.predict(X_test_b))

print('Iteration comparison:')
print(f'  Baseline (3 raw features):            {baseline_acc:.2%}')
print(f'  Engineered (5 features incl. derived): {test_acc:.2%}')
print()
if test_acc > baseline_acc:
    diff = test_acc - baseline_acc
    print(f'Feature engineering improved accuracy by {diff:.2%}.')
else:
    print('Feature engineering did not help in this case - raw features were already sufficient.')
print()
print('This is iteration: compare approaches, keep what works.')

In [ ]:
# Final: predict on 3 new students the system has never seen
new_students = pd.DataFrame({
    'study_hours':    [1.0,  7.5,  4.0 ],
    'attendance_pct': [45.0, 92.0, 70.0],
    'prev_score':     [30,   85,   60  ],
})
new_students['attended_enough'] = (new_students['attendance_pct'] >= 75).astype(int)
new_students['effort_score']    = new_students['study_hours'] * new_students['prev_score'] / 100

preds = model.predict(new_students[FEATURES])
probs = model.predict_proba(new_students[FEATURES])[:, 1]  # probability of passing

print('Predictions for 3 new students:')
for i in range(len(new_students)):
    row    = new_students.iloc[i]
    result = 'PASS' if preds[i] else 'FAIL'
    print(f'  Student {i+1}: study={row.study_hours}h, attendance={row.attendance_pct}%, prev={row.prev_score}')
    print(f'  Prediction: {result} (confidence: {probs[i]:.0%})')
    print()

---
## Summary

We followed the complete ML workflow:

| Step | What we did |
|---|---|
| 1. Define problem | Binary classification: predict WAEC pass/fail |
| 2. Prepare data | EDA, feature engineering, feature selection |
| 3. Split data | 80/20 train/test split - test set locked |
| 4. Train model | `DecisionTreeClassifier.fit(X_train, y_train)` |
| 5. Evaluate | Accuracy, classification report, feature importance |
| Iterate | Compared baseline vs engineered features |

**Next - Topic 4:** Overfitting & Underfitting. We deliberately break the model and learn to diagnose and fix it.